# ECG Classification - Advanced PyTorch CNN1D (v6)

## Overview
Advanced CNN1D model for ECG beat classification using MIT-BIH database.

## Key Features
- **CNN1D Architecture**: Multi-layer with residual connections
- **RR Interval Feature**: Beat duration as additional input
- **Anti-Overfitting**: Dropout, BatchNorm, Weight Decay, Early Stopping, ReduceLROnPlateau
- **Data Leakage Prevention**: Split FIRST, then normalize
- **ONNX Export**: For lightweight deployment

## Data Leakage Prevention
```
1. Split data FIRST (before any preprocessing)
2. Fit scaler ONLY on training data
3. Transform val/test using training statistics
```

## STEP 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, davies_bouldin_score
)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)

## STEP 2: Load Dataset

In [ ]:
import os

KAGGLE_PATH = '/kaggle/input/mitbih-dataset/mitbih_beats_dataset.csv'
LOCAL_PATH = '../dataset/mitbih_beats_dataset.csv'
ALT_PATH = 'mitbih_beats_dataset.csv'

if os.path.exists(KAGGLE_PATH):
    df = pd.read_csv(KAGGLE_PATH)
    print(f'Loaded from Kaggle: {KAGGLE_PATH}')
elif os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local: {LOCAL_PATH}')
elif os.path.exists(ALT_PATH):
    df = pd.read_csv(ALT_PATH)
    print(f'Loaded from: {ALT_PATH}')
else:
    raise FileNotFoundError('Dataset not found! Run create-mitbih-dataset.ipynb first.')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()[:5]}... + {df.columns.tolist()[-3:]}')
print(f'Label distribution:\n{df["label"].value_counts()}')

## STEP 3: EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Label (0=Normal, 1=Abnormal)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Normal', 'Abnormal'], rotation=0)

df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['green', 'red'])
axes[1].set_title('Label Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## STEP 4: Data Preprocessing - NO DATA LEAKAGE

**CRITICAL**: Split FIRST, then normalize!

In [ ]:
sample_cols = [c for c in df.columns if c.startswith('sample_')]
has_rr = 'rr_interval' in df.columns

if has_rr:
    X_samples = df[sample_cols].values
    X_rr = df['rr_interval'].values.reshape(-1, 1)
    print(f'Using RR interval as additional feature')
else:
    X_samples = df.drop('label', axis=1).values
    X_rr = None
    print(f'No RR interval, using samples only')

y = df['label'].values

print(f'Samples shape: {X_samples.shape}')
print(f'Labels shape: {y.shape}')

# SPLIT FIRST
if has_rr:
    X_samples_train, X_samples_temp, X_rr_train, X_rr_temp, y_train, y_temp = train_test_split(
        X_samples, X_rr, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    X_samples_val, X_samples_test, X_rr_val, X_rr_test, y_val, y_test = train_test_split(
        X_samples_temp, X_rr_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
    )
else:
    X_samples_train, X_samples_temp, y_train, y_temp = train_test_split(
        X_samples, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    X_samples_val, X_samples_test, y_val, y_test = train_test_split(
        X_samples_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
    )
    X_rr_train = X_rr_val = X_rr_test = None

print(f'Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}')

# NORMALIZE AFTER SPLIT - FIT ONLY ON TRAINING DATA
scaler_samples = StandardScaler()
X_samples_train_norm = scaler_samples.fit_transform(X_samples_train)
X_samples_val_norm = scaler_samples.transform(X_samples_val)
X_samples_test_norm = scaler_samples.transform(X_samples_test)

if has_rr:
    scaler_rr = StandardScaler()
    X_rr_train_norm = scaler_rr.fit_transform(X_rr_train)
    X_rr_val_norm = scaler_rr.transform(X_rr_val)
    X_rr_test_norm = scaler_rr.transform(X_rr_test)
else:
    scaler_rr = None

print(f'Normalization (training stats only):')
print(f'  Mean: {X_samples_train_norm.mean():.6f}, Std: {X_samples_train_norm.std():.6f}')

# Reshape for CNN1D: (batch, channels, length)
X_train_cnn = X_samples_train_norm.reshape(-1, 1, 188)
X_val_cnn = X_samples_val_norm.reshape(-1, 1, 188)
X_test_cnn = X_samples_test_norm.reshape(-1, 1, 188)

print(f'CNN input shape: {X_train_cnn.shape}')

## STEP 5: PyTorch Dataset

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X_cnn, X_rr, y):
        self.X_cnn = torch.FloatTensor(X_cnn)
        self.X_rr = torch.FloatTensor(X_rr) if X_rr is not None else None
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        if self.X_rr is not None:
            return self.X_cnn[idx], self.X_rr[idx], self.y[idx]
        return self.X_cnn[idx], self.y[idx]

train_dataset = ECGDataset(X_train_cnn, X_rr_train_norm if has_rr else None, y_train)
val_dataset = ECGDataset(X_val_cnn, X_rr_val_norm if has_rr else None, y_val)
test_dataset = ECGDataset(X_test_cnn, X_rr_test_norm if has_rr else None, y_test)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

## STEP 6: Advanced CNN1D Model

**Anti-Overfitting Features:**
- Dropout (0.5)
- BatchNorm after each conv layer
- Residual connections
- Separate RR interval input branch

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, stride=1, dropout=0.3):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride, padding)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, 1, padding)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride),
                nn.BatchNorm1d(out_ch)
            )
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


class AdvancedCNN1D(nn.Module):
    def __init__(self, use_rr=True, dropout=0.5):
        super().__init__()
        self.use_rr = use_rr
        
        # CNN backbone
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout * 0.5)
        )
        
        self.res1 = ResidualBlock(32, 64, dropout=dropout * 0.5)
        self.pool1 = nn.MaxPool1d(2)
        
        self.res2 = ResidualBlock(64, 128, dropout=dropout * 0.7)
        self.pool2 = nn.MaxPool1d(2)
        
        self.res3 = ResidualBlock(128, 256, dropout=dropout)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Feature dimension after CNN
        cnn_out = 256
        
        # RR interval branch
        if use_rr:
            self.rr_branch = nn.Sequential(
                nn.Linear(1, 16),
                nn.ReLU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(16, 32),
                nn.ReLU()
            )
            fc_in = cnn_out + 32
        else:
            fc_in = cnn_out
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(fc_in, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.8),
            nn.Linear(64, 2)
        )
    
    def forward(self, x_cnn, x_rr=None):
        # CNN path
        x = self.conv1(x_cnn)
        x = self.pool1(self.res1(x))
        x = self.pool2(self.res2(x))
        x = self.global_pool(self.res3(x))
        x = x.squeeze(-1)
        
        # Combine with RR if available
        if self.use_rr and x_rr is not None:
            rr_features = self.rr_branch(x_rr)
            x = torch.cat([x, rr_features], dim=1)
        
        return self.classifier(x)

model = AdvancedCNN1D(use_rr=has_rr, dropout=0.5).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## STEP 7: Training Configuration

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights}')

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-3  # Strong L2 regularization
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6, verbose=True
)

print('Training config: AdamW (lr=0.001, wd=1e-3), ReduceLROnPlateau')

## STEP 8: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, has_rr):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for batch in loader:
        if has_rr:
            x_cnn, x_rr, labels = batch
            x_cnn, x_rr, labels = x_cnn.to(device), x_rr.to(device), labels.to(device)
        else:
            x_cnn, labels = batch
            x_cnn, labels = x_cnn.to(device), labels.to(device)
            x_rr = None
        
        optimizer.zero_grad()
        outputs = model(x_cnn, x_rr)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    
    return total_loss / len(loader), correct / total


def validate_epoch(model, loader, criterion, device, has_rr):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for batch in loader:
            if has_rr:
                x_cnn, x_rr, labels = batch
                x_cnn, x_rr, labels = x_cnn.to(device), x_rr.to(device), labels.to(device)
            else:
                x_cnn, labels = batch
                x_cnn, labels = x_cnn.to(device), labels.to(device)
                x_rr = None
            
            outputs = model(x_cnn, x_rr)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()
    
    return total_loss / len(loader), correct / total

print('Training functions defined')

## STEP 9: Training Loop

In [ ]:
NUM_EPOCHS = 100
PATIENCE = 15
best_val_loss = float('inf')
patience_counter = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

print('Starting Training...')
print('=' * 70)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, has_rr)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device, has_rr)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']
    
    if (epoch + 1) % 5 == 0 or epoch < 5:
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {lr:.6f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, 'ecg_cnn_v6_pytorch_best.pth')
        if (epoch + 1) % 5 == 0:
            print(f'  -> New best model (val_loss: {val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

checkpoint = torch.load('ecg_cnn_v6_pytorch_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Best model from epoch {checkpoint["epoch"]+1}, val_acc: {checkpoint["val_acc"]:.4f}')

## STEP 10: Training Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='Train Acc')
ax2.plot(val_accs, label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history_v6.png', dpi=150)
plt.show()

## STEP 11: Comprehensive Evaluation

In [ ]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        if has_rr:
            x_cnn, x_rr, labels = batch
            x_cnn, x_rr = x_cnn.to(device), x_rr.to(device)
        else:
            x_cnn, labels = batch
            x_cnn = x_cnn.to(device)
            x_rr = None
        
        outputs = model(x_cnn, x_rr)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)
try:
    auc_roc = roc_auc_score(all_labels, all_probs)
except Exception:
    auc_roc = 0
try:
    logloss = log_loss(all_labels, all_probs)
except Exception:
    logloss = 0

mae = mean_absolute_error(all_labels, all_preds)
mse = mean_squared_error(all_labels, all_preds)
rmse = np.sqrt(mse)

print('=' * 70)
print('COMPREHENSIVE MODEL EVALUATION - Advanced CNN1D (v6-pytorch)')
print('=' * 70)
print('\nCATEGORY 1: BASIC CLASSIFICATION METRICS')
print('=' * 70)
print(f'1. ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'2. PRECISION: {precision:.4f}')
print(f'3. RECALL: {recall:.4f}')
print(f'4. F1 SCORE: {f1:.4f}')
print(f'5. LOG LOSS: {logloss:.4f}')
print('\nCATEGORY 2: ROC CURVE AND AUC METRICS')
print('=' * 70)
print(f'6. AUC-ROC: {auc_roc:.4f}')
print('\nCATEGORY 3: ERROR METRICS')
print('=' * 70)
print(f'8a. MAE: {mae:.6f}')
print(f'8b. MSE: {mse:.6f}')
print(f'8c. RMSE: {rmse:.6f}')

print('\n' + '=' * 70)
print('CLASSIFICATION REPORT')
print('=' * 70)
print(classification_report(all_labels, all_preds, target_names=['Normal (0)', 'Abnormal (1)']))

print('\nSUMMARY')
print('=' * 70)
print(f'Accuracy:    {accuracy:.4f}')
print(f'Precision:   {precision:.4f}')
print(f'Recall:      {recall:.4f}')
print(f'F1 Score:    {f1:.4f}')
print(f'AUC-ROC:     {auc_roc:.4f}')

## STEP 12: Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Normal', 'Abnormal'],
       yticklabels=['Normal', 'Abnormal'],
       title='Confusion Matrix', ylabel='True', xlabel='Predicted')
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.savefig('confusion_matrix_v6.png', dpi=150)
plt.show()

## STEP 13: Save Model and Scaler

In [ ]:
import joblib

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {'use_rr': has_rr, 'dropout': 0.5},
    'accuracy': accuracy,
}, 'ecg_cnn_v6_pytorch_final.pth')
print('Model saved: ecg_cnn_v6_pytorch_final.pth')

joblib.dump(scaler_samples, 'scaler_v6_pytorch.pkl')
print('Scaler saved: scaler_v6_pytorch.pkl')

if has_rr and scaler_rr:
    joblib.dump(scaler_rr, 'scaler_rr_v6_pytorch.pkl')
    print('RR Scaler saved: scaler_rr_v6_pytorch.pkl')

## STEP 14: Export to ONNX

In [ ]:
print('Exporting to ONNX...')

model.eval()
model_cpu = model.cpu()

# Create wrapper for ONNX export (CNN only, no RR for simpler deployment)
class ONNXWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.conv1 = base_model.conv1
        self.res1 = base_model.res1
        self.pool1 = base_model.pool1
        self.res2 = base_model.res2
        self.pool2 = base_model.pool2
        self.res3 = base_model.res3
        self.global_pool = base_model.global_pool
        # Simplified classifier (CNN features only)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(self.res1(x))
        x = self.pool2(self.res2(x))
        x = self.global_pool(self.res3(x))
        x = x.squeeze(-1)
        return self.classifier(x)

# For simpler deployment, create CNN-only version
dummy_input = torch.randn(1, 1, 188)
onnx_path = 'ecg_cnn_v6_pytorch_final.onnx'

try:
    torch.onnx.export(
        model_cpu,
        (dummy_input, torch.randn(1, 1)) if has_rr else dummy_input,
        onnx_path,
        export_params=True,
        opset_version=13,
        input_names=['ecg_input', 'rr_input'] if has_rr else ['input'],
        output_names=['output'],
        dynamic_axes={'ecg_input' if has_rr else 'input': {0: 'batch'}, 'output': {0: 'batch'}}
    )
    print(f'ONNX exported: {onnx_path}')
    
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)
    print('ONNX model verified successfully')
except Exception as e:
    print(f'ONNX export error: {e}')
    print('PyTorch model is available for manual export')

## STEP 15: Summary

In [ ]:
print('=' * 70)
print('TRAINING COMPLETE - Advanced CNN1D (v6)')
print('=' * 70)
print(f'\nAnti-Overfitting Techniques:')
print('  - Dropout: 0.5 (progressive)')
print('  - BatchNorm: After each conv/FC layer')
print('  - Weight Decay: 1e-3 (L2)')
print('  - Early Stopping: Patience 15')
print('  - ReduceLROnPlateau: Factor 0.5, Patience 5')
print('  - Gradient Clipping: max_norm=1.0')
print('  - Residual Connections')
print(f'\nData Leakage Prevention:')
print('  - Split FIRST, then normalize')
print('  - Scaler fit on training data ONLY')
print(f'\nFinal Metrics:')
print(f'  Accuracy: {accuracy:.4f}')
print(f'  F1 Score: {f1:.4f}')
print(f'  AUC-ROC:  {auc_roc:.4f}')
print(f'\nFiles Created:')
print('  - ecg_cnn_v6_pytorch_final.pth')
print('  - ecg_cnn_v6_pytorch_final.onnx')
print('  - scaler_v6_pytorch.pkl')
if has_rr:
    print('  - scaler_rr_v6_pytorch.pkl')
print('\nNext: Test on record 119 (excluded from training)')